In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting"

!pip install -U kaggle mlflow dagshub

Mounted at /content/drive
/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import zipfile
import glob

data_dir = "data"
required_files = ["train.csv", "test.csv", "stores.csv", "features.csv"]

files_exist = os.path.exists(data_dir) and all(
    os.path.exists(os.path.join(data_dir, f)) for f in required_files
)

if files_exist:
    print("Dataset is already downloaded and extracted in the 'data' folder. Skipping download!")
else:
    print("Dataset not found. Starting download...")

    os.environ['KAGGLE_API_TOKEN'] = "KGAT_caf00baa57d3f767dc15d31b24d4e730"

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting

    os.makedirs(data_dir, exist_ok=True)
    main_zip = "walmart-recruiting-store-sales-forecasting.zip"
    if os.path.exists(main_zip):
        with zipfile.ZipFile(main_zip, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(main_zip)
    else:
        print("The zip file wasn't found. Double check your Kaggle Token expiration status!")

    inner_zips = glob.glob(os.path.join(data_dir, "*.zip"))
    for file in inner_zips:
        with zipfile.ZipFile(file, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(file)

    print("Success! All datasets downloaded and extracted:")
    print(os.listdir(data_dir))

Dataset is already downloaded and extracted in the 'data' folder. Skipping download!


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import dagshub
import mlflow.xgboost
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.features.preprocessing import merge_and_enrich
from src.evaluation.metrics import calculate_wmae

dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)

train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

mlflow.set_experiment("XGBoost_Training_V1")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=986aa4b8-0b1d-483c-bee5-e5d4b8a3089b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=00bbf783241e069b77b990e5ea64789ad206dae88db577b84ba7a1a2a739da6f




Accessing as slosa23

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/37bfc13473bd4295a38087d01a7ddd0f', creation_time=1782641744510, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782641744510, lifecycle_stage='active', name='XGBoost_Training_V1', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    df_clean = merge_and_enrich(train_raw, stores, features)

    mlflow.log_param("markdown_imputation", "filled_with_zero")
    mlflow.log_param("missing_cpi_imputation", "median")

    mlflow.log_text("Train split limit: 2012-01-01", "split_strategy.txt")

    train_mask = df_clean['Date'] < '2012-01-01'
    val_mask = df_clean['Date'] >= '2012-01-01'

    train_data = df_clean[train_mask]
    val_data = df_clean[val_mask]

    print("Cleaning run completed and logged successfully!")

Cleaning run completed and logged successfully!
🏃 View run XGBoost_Cleaning at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/4ce019784c7c49e1985966e3379347ea
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1


In [ ]:
with mlflow.start_run(run_name="XGBoost_Pipeline_Fitting"):

    num_features = ['Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
                    'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment',
                    'Year', 'Month', 'Week', 'Day', 'IsHoliday']
    cat_features = ['Type']

    X_train = train_data[num_features + cat_features]
    y_train = train_data['Weekly_Sales']

    X_val = val_data[num_features + cat_features]
    y_val = val_data['Weekly_Sales']
    is_holiday_val = val_data['IsHoliday']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
            ('num', 'passthrough', num_features)
        ]
    )

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1))
    ])

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)

    print("Training XGBoost Pipeline...")
    model_pipeline.fit(X_train, y_train)

    preds = model_pipeline.predict(X_val)
    val_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"Validation WMAE: {val_wmae:.2f}")

    mlflow.log_metric("WMAE", val_wmae)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor'
    ]

    mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        name="model",
        skops_trusted_types=trusted_types
    )

    print("Pipeline successfully registered and logged to MLflow!")

Training XGBoost Pipeline...
Validation WMAE: 14274.07
Pipeline successfully registered and logged to MLflow!
🏃 View run XGBoost_Pipeline_Fitting at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/d563a230d0d1413ca4d59037ee8a81ec
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1


In [ ]:
with mlflow.start_run(run_name="XGBoost_Hyperparameter_Tuning"):

    num_features = ['Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
                    'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment',
                    'Year', 'Month', 'Week', 'Day', 'IsHoliday']
    cat_features = ['Type']

    X_train = train_data[num_features + cat_features]
    y_train = train_data['Weekly_Sales']
    X_val = val_data[num_features + cat_features]
    y_val = val_data['Weekly_Sales']
    is_holiday_val = val_data['IsHoliday']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
            ('num', 'passthrough', num_features)
        ]
    )

    tuned_xgb = xgb.XGBRegressor(
        n_estimators=250,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    tuned_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', tuned_xgb)
    ])

    mlflow.log_param("n_estimators", 250)
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)

    print("Training Tuned XGBoost Pipeline (This may take a minute longer)...")
    tuned_pipeline.fit(X_train, y_train)

    tuned_preds = tuned_pipeline.predict(X_val)
    tuned_wmae = calculate_wmae(y_val, tuned_preds, is_holiday_val)
    print(f"Tuned Validation WMAE: {tuned_wmae:.2f}")

    mlflow.log_metric("WMAE", tuned_wmae)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor'
    ]

    mlflow.sklearn.log_model(
        sk_model=tuned_pipeline,
        name="tuned_model",
        skops_trusted_types=trusted_types
    )

    print("Tuned model successfully logged to MLflow!")

Training Tuned XGBoost Pipeline (This may take a minute longer)...
Tuned Validation WMAE: 14091.24
Tuned model successfully logged to MLflow!
🏃 View run XGBoost_Hyperparameter_Tuning at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/59fce9f3497f4fa389916fcc0b3e4344
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1


In [ ]:
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from src.features.preprocessing import get_model_ready_data
from src.evaluation.metrics import calculate_wmae

mlflow.sklearn.autolog(max_tuning_runs=None, log_models=True)

with mlflow.start_run(run_name="XGBoost_Grid_Search_Parent"):

    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date='2012-01-01'
    )

    base_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', xgb.XGBRegressor(random_state=42, n_jobs=-1))
    ])

    param_grid = {
        'regressor__max_depth': [5, 7],
        'regressor__learning_rate': [0.1, 0.05],
        'regressor__n_estimators': [150]
    }

    tscv = TimeSeriesSplit(n_splits=3)

    grid_search = GridSearchCV(
        estimator=base_pipeline,
        param_grid=param_grid,
        cv=tscv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    print("Starting Grid Search")
    grid_search.fit(X_train, y_train)

    best_pipeline = grid_search.best_estimator_
    preds = best_pipeline.predict(X_val)
    final_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"Champion Model Validation WMAE: {final_wmae:.2f}")

    mlflow.log_metric("Champion_WMAE", final_wmae)

    mlflow.sklearn.autolog(disable=True)

    print("Grid search finished")

Starting Grid Search


2026/06/28 10:38:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 3 folds for each of 4 candidates, totalling 12 fits


2026/06/28 10:38:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/28 10:38:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely

🏃 View run intelligent-ray-83 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/f9c5dc307cfb444e8167d127da77de12
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1
🏃 View run gentle-ant-263 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/71847d3511a845e5b97830f67e8f5060
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1
🏃 View run calm-squirrel-497 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/be7e1910d8d94e848033b96dba97d4f8
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1
🏃 View run thoughtful-tern-421 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/96dcc0fe6cf849f1b4c926301bf72b1b
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1
Champion Mod

In [ ]:
import yaml
import mlflow
import mlflow.sklearn
from src.features.preprocessing import get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.evaluation.metrics import calculate_wmae

with open("configs/xgboost_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
model_params = config['model']['params']
split_date = config['data']['split_date']

with mlflow.start_run(run_name=model_name):

    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    pipeline = create_xgboost_pipeline(preprocessor, model_params)

    mlflow.log_params(model_params)
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_val)
    production_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"Optimized Production Run WMAE: {production_wmae:.2f}")

    mlflow.log_metric("WMAE", production_wmae)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor'
    ]
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="production_champion_model",
        skops_trusted_types=trusted_types
    )

    print("Pipeline run completely tracked and registered to DagsHub!")

Optimized Production Run WMAE: 14113.72
Pipeline run completely tracked and registered to DagsHub!
🏃 View run XGBoost_Structured_Production at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/1d140b52a7b54240b085ecef72d05dd5
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1


In [ ]:
import yaml
import mlflow
import mlflow.sklearn
from src.features.preprocessing import get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.evaluation.metrics import calculate_wmae

with open("configs/xgboost_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
model_params = config['model']['params']
split_date = config['data']['split_date']

with mlflow.start_run(run_name=f"{model_name}_Production_Match"):

    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    pipeline = create_xgboost_pipeline(preprocessor, model_params)

    mlflow.log_params(model_params)
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_val)
    production_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"Confirmed Production WMAE: {production_wmae:.2f}")

    mlflow.log_metric("WMAE", production_wmae)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor'
    ]
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        skops_trusted_types=trusted_types,
        registered_model_name=model_name
    )

    print(f"'{model_name}' version logged and registered to DagsHub!")

2026/06/30 22:24:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Confirmed Production WMAE: 14113.72


Successfully registered model 'Walmart_XGBoost_Baseline'.
2026/06/30 22:24:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_XGBoost_Baseline, version 1
Created version '1' of model 'Walmart_XGBoost_Baseline'.


'Walmart_XGBoost_Baseline' version logged and registered to DagsHub!
🏃 View run Walmart_XGBoost_Baseline_Production_Match at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/c5662c3d50d24dae94367be8b56e88ac
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1


In [6]:
import yaml
import mlflow
import mlflow.sklearn
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from src.features.preprocessing import get_model_ready_data
from src.evaluation.metrics import calculate_wmae

with open("configs/xgboost_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
model_params = config['model']['params']
split_date = config['data']['split_date']

with mlflow.start_run(run_name=model_name):
    print("Extracting freshly engineered data shapes...")

    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    print(f"Initializing {model_name} architecture...")
    xgb_regressor = XGBRegressor(**model_params)

    upgraded_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', xgb_regressor)
    ])

    mlflow.log_params(model_params)

    print("Fitting upgraded pipeline across new feature coordinates")
    upgraded_pipeline.fit(X_train, y_train)

    print("Evaluating prediction array performance")
    preds = upgraded_pipeline.predict(X_val)
    upgraded_wmae = calculate_wmae(y_val, preds, is_holiday_val)
    print(f"Upgraded Standalone Tree Baseline WMAE: {upgraded_wmae:.2f}")

    mlflow.log_metric("WMAE", upgraded_wmae)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor',
        'numpy.dtype'
    ]

    print(f"Uploading asset structure")
    mlflow.sklearn.log_model(
        sk_model=upgraded_pipeline,
        name="model",
        skops_trusted_types=trusted_types,
        registered_model_name="Walmart_XGBoost_Baseline"
    )

    print(f"Complete! Native pipeline structure is registered.")

Extracting freshly engineered data shapes...
Initializing Walmart_XGBoost_Upgraded_Baseline architecture...
Fitting upgraded pipeline across new feature coordinates
Evaluating prediction array performance
Upgraded Standalone Tree Baseline WMAE: 3465.01
Uploading asset structure


Registered model 'Walmart_XGBoost_Baseline' already exists. Creating a new version of this model...
2026/07/12 13:19:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_XGBoost_Baseline, version 2
Created version '2' of model 'Walmart_XGBoost_Baseline'.


Complete! Native pipeline structure is registered.
🏃 View run Walmart_XGBoost_Upgraded_Baseline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/f82737f877264103b0ec6f5f72736eeb
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/1
